# Day 4 — Update uncertainty and generate data

All data are synthetic. Run from top to bottom. Work in pairs and pause after each result to explain its meaning. Use TEACHING_GUIDE.md and TASK_CARDS.md for timing. Optional sections are marked. Numerical outputs are examples, not evidence about real operations.

## 1. A probability model for defects

Let p be an unknown defect probability. Prior Beta(2,18) has mean 0.10. Observing 8 defects in 100 independent inspected items yields Beta(10,110). This assumes a stable rate and a representative sample. Posterior is proportional to likelihood times prior.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import beta
rng=np.random.default_rng(44)
a,b=2,18
n,k=100,8
post_a,post_b=a+k,b+n-k
lo,hi=beta.ppf([.025,.975],post_a,post_b)
print('Prior mean:',a/(a+b))
print('Posterior parameters:',post_a,post_b)
print('Posterior mean:',post_a/(post_a+post_b))
print('95% equal-tailed credible interval:',np.round([lo,hi],4))
print('Posterior probability p > 0.10:',round(beta.sf(.10,post_a,post_b),4))
grid=np.linspace(.001,.35,400)
plt.figure(figsize=(7,3));plt.plot(grid,beta.pdf(grid,a,b),label='Prior');plt.plot(grid,beta.pdf(grid,post_a,post_b),label='Posterior');plt.xlabel('Defect probability');plt.ylabel('Probability density');plt.legend();plt.tight_layout();plt.show()

Prior mean: 0.1
Posterior parameters: 10 110
Posterior mean: 0.08333333333333333
95% equal-tailed credible interval: [0.041  0.1387]
Posterior probability p > 0.10: 0.2375


## 2. MCMC: a real Metropolis sampler — guided demonstration

Direct Beta sampling is available, so MCMC is unnecessary for this problem; the exact posterior lets us check the sampler. Propose a symmetric Gaussian step in p, reject proposals outside (0,1), then accept according to the posterior-density ratio. Repeated states are legitimate. Never clip proposals to the boundary: that changes the proposal mechanism.

In [2]:
def metropolis(start,seed,steps=12000,proposal_sd=.025):
    random=np.random.default_rng(seed)
    p=start; chain=np.empty(steps); accepted=0
    for i in range(steps):
        proposal=p+random.normal(0,proposal_sd)
        if 0<proposal<1:
            log_ratio=beta.logpdf(proposal,post_a,post_b)-beta.logpdf(p,post_a,post_b)
            if np.log(random.uniform())<min(0,log_ratio):
                p=proposal;accepted+=1
        chain[i]=p
    return chain,accepted/steps
runs=[metropolis(start,seed) for start,seed in [(0.02,1),(0.35,2),(0.65,3),(0.9,4)]]
chains=np.array([run[0][2000:] for run in runs])
print('Acceptance rates:',np.round([run[1] for run in runs],3))
print('Retained-chain means:',np.round(chains.mean(axis=1),4))
print('Exact mean:',round(post_a/(post_a+post_b),4))
print('Pooled MCMC interval:',np.round(np.quantile(chains,[.025,.975]),4))
fig,axes=plt.subplots(1,2,figsize=(10,3))
for row in chains: axes[0].plot(row[:500],alpha=.6)
axes[0].set(xlabel='Retained iteration',ylabel='p',title='First 500 retained draws')
axes[1].hist(chains.ravel(),bins=40,density=True,alpha=.5);axes[1].plot(grid,beta.pdf(grid,post_a,post_b));axes[1].set(xlabel='p',title='Sample versus exact density')
plt.tight_layout();plt.show()
print('Lag-1 correlations:',np.round([np.corrcoef(row[:-1],row[1:])[0,1] for row in chains],3))

Acceptance rates: [0.707 0.696 0.7   0.7  ]
Retained-chain means: [0.0831 0.0842 0.0831 0.084 ]
Exact mean: 0.0833
Pooled MCMC interval: [0.0411 0.1388]
Lag-1 correlations: [0.777 0.786 0.783 0.781]


## 3. Posterior predictive simulation

Uncertainty about p and variability in a new batch are different. Draw p from the posterior, then draw the number of defects in 100 future items. This produces synthetic counts. A credible interval for p is not a predictive interval for a future defect count.

In [3]:
p_draws=rng.beta(post_a,post_b,10000)
future_counts=rng.binomial(100,p_draws)
print('Expected future defects in 100:',round(future_counts.mean(),2))
print('Central 95% simulated count interval:',np.quantile(future_counts,[.025,.975]))
print('Probability of more than 12 future defects:',round(np.mean(future_counts>12),3))
plt.figure(figsize=(7,3));plt.hist(future_counts,bins=np.arange(0,32)-.5);plt.xlabel('Defects in next 100 items');plt.ylabel('Simulation count');plt.tight_layout();plt.show()

Expected future defects in 100: 8.32
Central 95% simulated count interval: [ 2. 17.]
Probability of more than 12 future defects: 0.136


## 4. Generate synthetic service times

A fitted lognormal model produces positive synthetic durations. Fit in log space and sample. This is a simple generative statistical model, not a GAN. Matching a marginal mean does not preserve relationships, privacy or realism.

In [4]:
observed=rng.lognormal(mean=3,sigma=.35,size=150)
log_mu=np.log(observed).mean();log_sd=np.log(observed).std(ddof=1)
synthetic=rng.lognormal(log_mu,log_sd,150)
summary=pd.DataFrame({'observed':observed,'synthetic':synthetic}).describe().loc[['mean','std','min','50%','max']]
print(summary.round(2))
fig,ax=plt.subplots(figsize=(7,3));ax.hist(observed,bins=15,alpha=.5,label='Observed synthetic source');ax.hist(synthetic,bins=15,alpha=.5,label='Generated');ax.set_xlabel('Service minutes');ax.legend();plt.tight_layout();plt.show()

      observed  synthetic
mean     20.55      20.20
std       6.87       6.69
min       7.15       6.60
50%      19.19      18.98
max      47.34      42.92
